In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import requests
import json

In [19]:
csv_filename = 'monthly_sales_data.csv'
# Read the CSV file from the specified path
full_path = os.path.join(r'data', csv_filename)
df = pd.read_csv(full_path)
df.head()

,sku name,sku nbr,state territory code,sub class,region,sales units,month
0,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-12-01
1,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,-1.0,2026-03-01
2,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-02-01
3,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,3.0,2026-01-01
4,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-10-01


In [24]:
df_monthly_sales_data = df.copy()

In [26]:
df_monthly_sales_data = df_monthly_sales_data.rename(columns={
    'sku name': 'sku_name', 
    'sku nbr': 'sku_nbr', 
    'sales units': 'sales_units',
    'state territory code': 'state_territory_code',
    'sub class': 'sub_class',    
})
df_monthly_sales_data.head()

,sku_name,sku_nbr,state_territory_code,sub_class,region,sales_units,month
0,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-12-01
1,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,-1.0,2026-03-01
2,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-02-01
3,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,3.0,2026-01-01
4,"1-1/2"" ABS DOUBLE SANI TEE HXHXHXH",134633,AK,026-001-005-ABS FITTINGS,0012-PACIFIC NORTHWEST,5.0,2025-10-01


In [38]:
df_monthly_sales_data['sku_nbr'].nunique()

905

In [9]:
import os

print(os.path.abspath("config.json"))

with open("config.json", "r") as f:
    print(repr(f.read()))

c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\config.json
'{\n  "FRED_API_KEY": "8a8f9ab6bab1b068658f888aad3dce92"\n}'


In [32]:
API_KEY = "8a8f9ab6bab1b068658f888aad3dce92"
BASE_URL = "https://api.stlouisfed.org/fred/series/observations"


def get_fred_series(series_id):
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
    }
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.rename(columns={"date": "month", "value": series_id})
    return df


def add_transforms(df, col):
    df = df.copy()

    # raw
    df[f"{col}_level"] = df[col]

    # month-over-month percent change
    df[f"{col}_pct_1m"] = df[col].pct_change()

    # year-over-year percent change
    df[f"{col}_pct_12m"] = df[col].pct_change(12)

    # first difference
    df[f"{col}_diff_1m"] = df[col].diff()

    # rolling means
    df[f"{col}_roll3"] = df[col].rolling(3).mean()
    df[f"{col}_roll6"] = df[col].rolling(6).mean()

    return df


def add_lags(df, col_list, lags=(0, 1, 3, 6)):
    df = df.copy()
    for col in col_list:
        for lag in lags:
            df[f"{col}_lag{lag}"] = df[col].shift(lag)
    return df


# ----------------------------
# 1. Prepare monthly SKU sales
# ----------------------------
df = df_monthly_sales_data.copy()
df["month"] = pd.to_datetime(df["month"])

df_sales = (
    df.groupby(["sku_nbr", "month"], as_index=False)["sales_units"]
      .sum()
      .sort_values(["sku_nbr", "month"])
)

# Add transforms for sales within each SKU
df_sales["sales_level"] = df_sales["sales_units"]
df_sales["sales_pct_1m"] = df_sales.groupby("sku_nbr")["sales_units"].pct_change()
df_sales["sales_pct_12m"] = df_sales.groupby("sku_nbr")["sales_units"].pct_change(12)
df_sales["sales_diff_1m"] = df_sales.groupby("sku_nbr")["sales_units"].diff()
df_sales["sales_roll3"] = (
    df_sales.groupby("sku_nbr")["sales_units"]
    .transform(lambda s: s.rolling(3).mean())
)
df_sales["sales_roll6"] = (
    df_sales.groupby("sku_nbr")["sales_units"]
    .transform(lambda s: s.rolling(6).mean())
)

# ----------------------------
# 2. Pull FRED data
# ----------------------------
df_houst = get_fred_series("HOUST")
df_computsa = get_fred_series("COMPUTSA")

df_fred = df_houst.merge(df_computsa, on="month", how="outer").sort_values("month")

# Add transforms for FRED columns
for fred_col in ["HOUST", "COMPUTSA"]:
    df_fred = add_transforms(df_fred, fred_col)

fred_feature_cols = [c for c in df_fred.columns if c != "month"]

# Add lagged versions of all FRED transformed columns
df_fred = add_lags(df_fred, fred_feature_cols, lags=(0, 1, 3, 6))

# Keep only lagged feature columns
fred_lag_cols = [c for c in df_fred.columns if c != "month" and "_lag" in c]

# ----------------------------
# 3. Merge sales + FRED
# ----------------------------
df_model = df_sales.merge(df_fred[["month"] + fred_lag_cols], on="month", how="left")

# ----------------------------
# 4. Run correlation search
# ----------------------------
sales_features = [
    "sales_level",
    "sales_pct_1m",
    "sales_pct_12m",
    "sales_diff_1m",
    "sales_roll3",
    "sales_roll6",
]

results = []

for sku, g in df_model.groupby("sku_nbr"):
    for sales_col in sales_features:
        for fred_col in fred_lag_cols:
            temp = g[[sales_col, fred_col]].dropna()

            if len(temp) < 12:
                continue

            corr = temp[sales_col].corr(temp[fred_col])

            results.append({
                "sku_nbr": sku,
                "sales_feature": sales_col,
                "fred_feature": fred_col,
                "correlation": corr,
                "abs_correlation": abs(corr),
                "n_obs": len(temp),
            })

df_corr_results = pd.DataFrame(results)

# ----------------------------
# 5. Best result per SKU
# ----------------------------
df_best_per_sku = (
    df_corr_results
    .sort_values(["sku_nbr", "abs_correlation"], ascending=[True, False])
    .groupby("sku_nbr", as_index=False)
    .first()
)

print(df_best_per_sku.head(20))

c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2882: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2887: RuntimeWarning: invalid value encountered in dot
  c = dot(X, X_T.conj())
c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\_core\_methods.py:132: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


    sku_nbr  sales_feature           fred_feature  correlation  \
0    110241    sales_roll6    COMPUTSA_roll6_lag3     0.775823   
1    110269    sales_roll6  COMPUTSA_pct_12m_lag3     0.662000   
2    110630    sales_roll6    COMPUTSA_roll3_lag0    -0.606411   
3    110661    sales_roll6       HOUST_roll6_lag3     0.629597   
4    110689  sales_pct_12m    COMPUTSA_roll3_lag6     0.579756   
5    113053    sales_roll6    COMPUTSA_roll3_lag1    -0.540596   
6    120088    sales_roll6       HOUST_roll3_lag0    -0.470935   
7    134633    sales_roll3  COMPUTSA_pct_12m_lag3     0.646949   
8    134634    sales_roll3  COMPUTSA_pct_12m_lag3     0.658114   
9    142559  sales_pct_12m    COMPUTSA_roll3_lag0     0.696430   
10   144650    sales_roll6             HOUST_lag0    -0.442583   
11   144700    sales_roll6       HOUST_roll3_lag0    -0.527347   
12   144745  sales_pct_12m  COMPUTSA_pct_12m_lag0    -0.605345   
13   145587    sales_roll3       HOUST_roll6_lag3     0.643513   
14   14559

In [33]:
print(
    df_corr_results
    .sort_values("abs_correlation", ascending=False)
    .head(50)
)

           sku_nbr  sales_feature         fred_feature  correlation  \
267733  1011642227    sales_roll6  COMPUTSA_roll6_lag1    -0.978452   
267732  1011642227    sales_roll6  COMPUTSA_roll6_lag0    -0.975832   
187654      760407    sales_roll6  COMPUTSA_roll6_lag3     0.957093   
267729  1011642227    sales_roll6  COMPUTSA_roll3_lag1    -0.953121   
267730  1011642227    sales_roll6  COMPUTSA_roll3_lag3    -0.951251   
120732      394909    sales_roll6  COMPUTSA_roll6_lag0    -0.950348   
235253  1002347266    sales_roll6  COMPUTSA_roll6_lag1     0.950057   
267454  1011642044    sales_roll6  COMPUTSA_roll6_lag3    -0.947159   
267453  1011642044    sales_roll6  COMPUTSA_roll6_lag1    -0.940325   
226046  1000540457    sales_roll6     HOUST_roll6_lag3    -0.935558   
120728      394909    sales_roll6  COMPUTSA_roll3_lag0    -0.932042   
238221  1003179849    sales_roll6  COMPUTSA_roll6_lag1    -0.929830   
267450  1011642044    sales_roll6  COMPUTSA_roll3_lag3    -0.928899   
235250

In [40]:
def get_fred_series(series_id: str) -> pd.DataFrame:
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
    }

    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    return df.rename(columns={"date": "month", "value": series_id})


def add_fred_features(df_fred: pd.DataFrame, col: str, lags=(0, 1, 3, 6)) -> pd.DataFrame:
    df_fred = df_fred.copy()

    # Base transforms
    df_fred[f"{col}_level"] = df_fred[col]
    df_fred[f"{col}_pct_1m"] = df_fred[col].pct_change()
    df_fred[f"{col}_pct_12m"] = df_fred[col].pct_change(12)
    df_fred[f"{col}_roll3"] = df_fred[col].rolling(3).mean()

    base_cols = [
        f"{col}_level",
        f"{col}_pct_1m",
        f"{col}_pct_12m",
        f"{col}_roll3",
    ]

    # Lagged versions
    for base_col in base_cols:
        for lag in lags:
            df_fred[f"{base_col}_lag{lag}"] = df_fred[base_col].shift(lag)

    return df_fred


def prepare_sales_features(df_sales: pd.DataFrame) -> pd.DataFrame:
    df_sales = df_sales.copy()
    df_sales = df_sales.sort_values(["sku_nbr", "month"])

    df_sales["sales_level"] = df_sales["sales_units"]
    df_sales["sales_pct_1m"] = df_sales.groupby("sku_nbr")["sales_units"].pct_change()
    df_sales["sales_pct_12m"] = df_sales.groupby("sku_nbr")["sales_units"].pct_change(12)
    df_sales["sales_roll3"] = (
        df_sales.groupby("sku_nbr")["sales_units"]
        .transform(lambda s: s.rolling(3).mean())
    )

    return df_sales


def run_sku_correlation_analysis(
    df_monthly_sales_data: pd.DataFrame,
    min_obs: int = 18,
    lags=(0, 1, 3, 6),
):    
    #Prepare monthly sales  
    df = df_monthly_sales_data.copy()
    df["month"] = pd.to_datetime(df["month"])

    # normalize month to month start
    df["month"] = df["month"].dt.to_period("M").dt.to_timestamp()
    sku_lookup = (df[["sku_nbr", "sku_name", "sub_class", "region", "state_territory_code"]].drop_duplicates(subset=["sku_nbr"])    )
    df_sales = (df.groupby(["sku_nbr", "month"], as_index=False)["sales_units"].sum()    )
    df_sales = prepare_sales_features(df_sales)

    #Pull and prepare FRED
    df_houst = get_fred_series("HOUST")
    df_computsa = get_fred_series("COMPUTSA")
    df_fred = (df_houst.merge(df_computsa, on="month", how="outer").sort_values("month"))
    df_fred["month"] = df_fred["month"].dt.to_period("M").dt.to_timestamp()
    df_fred = add_fred_features(df_fred, "HOUST", lags=lags)
    df_fred = add_fred_features(df_fred, "COMPUTSA", lags=lags)

    fred_feature_cols = [
        c for c in df_fred.columns
        if c.startswith("HOUST_") or c.startswith("COMPUTSA_")
    ]
    
    # merge sales and FRED    
    df_model = df_sales.merge(
        df_fred[["month"] + fred_feature_cols],
        on="month",
        how="left",
    )

    sales_feature_cols = [
        "sales_level",
        "sales_pct_1m",
        "sales_pct_12m",
        "sales_roll3",
    ]
 
    # grid search    
    results = []

    for sku, g in df_model.groupby("sku_nbr"):
        for sales_col in sales_feature_cols:
            for fred_col in fred_feature_cols:
                temp = g[[sales_col, fred_col]].dropna()

                if len(temp) < min_obs:
                    continue

                # Avoid undefined corr when one side is constant
                if temp[sales_col].nunique() <= 1 or temp[fred_col].nunique() <= 1:
                    continue

                corr = temp[sales_col].corr(temp[fred_col])

                if pd.isna(corr):
                    continue

                results.append({
                    "sku_nbr": sku,
                    "sales_feature": sales_col,
                    "fred_feature": fred_col,
                    "correlation": corr,
                    "abs_correlation": abs(corr),
                    "n_obs": len(temp),
                })

    df_corr_results = pd.DataFrame(results)

    if df_corr_results.empty:
        raise ValueError("No valid correlations were computed. Check date alignment and data coverage.")

    #best correlation per SKU
    df_best_per_sku = (
        df_corr_results
        .sort_values(["sku_nbr", "abs_correlation"], ascending=[True, False])
        .groupby("sku_nbr", as_index=False)
        .first()
    )

    df_best_per_sku = df_best_per_sku.merge(sku_lookup, on="sku_nbr", how="left")    
    df_best_per_sku["fred_series"] = df_best_per_sku["fred_feature"].str.extract(r"^(HOUST|COMPUTSA)")[0]

    df_best_per_sku["fred_transform"] = df_best_per_sku["fred_feature"].str.extract(
        r"^(?:HOUST|COMPUTSA)_(level|pct_1m|pct_12m|roll3)"
    )[0]

    df_best_per_sku["lag_months"] = pd.to_numeric(
        df_best_per_sku["fred_feature"].str.extract(r"lag(\d+)")[0],
        errors="coerce"
    ).astype("Int64")

  
    # Summary tables  
    feature_summary = (df_best_per_sku.groupby(["fred_series", "fred_transform", "lag_months"], as_index=False).agg(
            sku_count=("sku_nbr", "count"),
            avg_corr=("correlation", "mean"),
            avg_abs_corr=("abs_correlation", "mean"),
            median_abs_corr=("abs_correlation", "median"),
        )
        .sort_values(["sku_count", "avg_abs_corr"], ascending=[False, False])
    )

    subclass_summary = (df_best_per_sku.groupby(["sub_class", "fred_series", "fred_transform", "lag_months"], as_index=False).agg(
            sku_count=("sku_nbr", "count"),
            avg_abs_corr=("abs_correlation", "mean"),
        )
        .sort_values(["sub_class", "sku_count", "avg_abs_corr"], ascending=[True, False, False])
    )

    region_summary = (df_best_per_sku.groupby(["region", "fred_series", "fred_transform", "lag_months"], as_index=False).agg(
            sku_count=("sku_nbr", "count"),
            avg_abs_corr=("abs_correlation", "mean"),
        )
        .sort_values(["region", "sku_count", "avg_abs_corr"], ascending=[True, False, False])
    )

    return df_corr_results, df_best_per_sku, feature_summary, subclass_summary, region_summary

df_corr_results, df_best_per_sku, feature_summary, subclass_summary, region_summary = run_sku_correlation_analysis(
    df_monthly_sales_data=df_monthly_sales_data,
    min_obs=18,
    lags=(0, 1, 3, 6),
)

print(df_best_per_sku.head(20))
print(feature_summary.head(20))
print(df_corr_results.sort_values("abs_correlation", ascending=False).head(20))

c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2882: RuntimeWarning: invalid value encountered in subtract
  X -= avg[:, None]
c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2887: RuntimeWarning: invalid value encountered in dot
  c = dot(X, X_T.conj())
c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\_core\_methods.py:132: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)


    sku_nbr  sales_feature           fred_feature  correlation  \
0    110241    sales_roll3    COMPUTSA_roll3_lag3     0.646332   
1    110269  sales_pct_12m    COMPUTSA_roll3_lag1     0.600660   
2    110630    sales_roll3       HOUST_roll3_lag3     0.574733   
3    110661    sales_roll3       HOUST_roll3_lag3     0.482470   
4    110689  sales_pct_12m    COMPUTSA_roll3_lag6     0.579756   
5    113053    sales_roll3            HOUST_roll3    -0.511099   
6    120088  sales_pct_12m     HOUST_pct_12m_lag1     0.427168   
7    134633    sales_roll3  COMPUTSA_pct_12m_lag3     0.646949   
8    134634    sales_roll3  COMPUTSA_pct_12m_lag3     0.658114   
9    142559  sales_pct_12m         COMPUTSA_roll3     0.696430   
10   144650   sales_pct_1m   COMPUTSA_pct_1m_lag1     0.419363   
11   144700   sales_pct_1m   COMPUTSA_pct_1m_lag1     0.386502   
12   144745  sales_pct_12m       COMPUTSA_pct_12m    -0.605345   
13   145587    sales_roll3  COMPUTSA_pct_12m_lag1     0.497463   
14   14559

In [42]:
df_final_coefficients = df_best_per_sku[
    ["sku_nbr", "sku_name", "correlation", "abs_correlation", "sales_feature", "fred_feature", "n_obs"]
].copy()

print(df_final_coefficients)

        sku_nbr                            sku_name  correlation  \
0        110241             4"X1-1/2" DWV WYE HXHXH     0.646332   
1        110269        4"X1-1/2" DWV SANI TEE HXHXH     0.600660   
2        110630  2" DWV SOIL PIPE ADAPTER HX NO HUB     0.574733   
3        110661  3" DWV SOIL PIPE ADAPTER HX NO HUB     0.482470   
4        110689  4" DWV SOIL PIPE ADAPTER HX NO HUB     0.579756   
..          ...                                 ...          ...   
773  1006145870           2 DWV FLUSH CLEANOUT PLUG    -0.360516   
774  1007427211         1/2 SCH 40 DEEP SOCKET CPLG     0.788857   
775  1007540445          1" SCH 40 DEEP SOCKET CPLG     0.609280   
776  1011642044        1 1/2 IN. ABS EXT SWVL ADPTR    -0.872664   
777  1011642227       1 1/2  ABS SWVL PLUG ADA WSHR    -0.922693   

     abs_correlation  sales_feature         fred_feature  n_obs  
0           0.646332    sales_roll3  COMPUTSA_roll3_lag3     35  
1           0.600660  sales_pct_12m  COMPUTSA_roll3

In [ ]:
# df_final_coefficients.to_csv("data/correlation_coefficients.csv")

In [45]:
def get_fred_series(series_id: str) -> pd.DataFrame:
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
    }
    response = requests.get(BASE_URL, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df.rename(columns={"date": "month", "value": series_id})


def simple_regression_beta(x: pd.Series, y: pd.Series):
    temp = pd.concat([x, y], axis=1).dropna()

    if len(temp) < 12:
        return np.nan, np.nan, len(temp)

    x_clean = temp.iloc[:, 0]
    y_clean = temp.iloc[:, 1]

    if x_clean.nunique() <= 1 or y_clean.nunique() <= 1:
        return np.nan, np.nan, len(temp)

    beta = np.cov(x_clean, y_clean, ddof=1)[0, 1] / np.var(x_clean, ddof=1)
    alpha = y_clean.mean() - beta * x_clean.mean()

    return beta, alpha, len(temp)


# -----------------------
# 1. Prepare your sales
# -----------------------
df = df_monthly_sales_data.copy()
df["month"] = pd.to_datetime(df["month"]).dt.to_period("M").dt.to_timestamp()

df_sales = (
    df.groupby(["sku_nbr", "month"], as_index=False)["sales_units"]
    .sum()
)

sku_lookup = (
    df[["sku_nbr", "sku_name", "sub_class", "region", "state_territory_code"]]
    .drop_duplicates(subset=["sku_nbr"])
)

# -----------------------
# 2. Pull FRED data
# -----------------------
df_houst = get_fred_series("HOUST")
df_computsa = get_fred_series("COMPUTSA")

df_fred = (
    df_houst.merge(df_computsa, on="month", how="outer")
    .sort_values("month")
)

df_fred["month"] = df_fred["month"].dt.to_period("M").dt.to_timestamp()

# -----------------------
# 3. Merge
# -----------------------
df_model = df_sales.merge(df_fred, on="month", how="left")

# -----------------------
# 4. Per-SKU betas
# -----------------------
results = []

for sku, g in df_model.groupby("sku_nbr"):
    beta_computsa, alpha_computsa, n1 = simple_regression_beta(
        g["COMPUTSA"], g["sales_units"]
    )
    beta_houst, alpha_houst, n2 = simple_regression_beta(
        g["HOUST"], g["sales_units"]
    )

    corr_computsa = g["sales_units"].corr(g["COMPUTSA"])
    corr_houst = g["sales_units"].corr(g["HOUST"])

    results.append({
        "sku_nbr": sku,
        "beta_computsa": beta_computsa,
        "alpha_computsa": alpha_computsa,
        "corr_computsa": corr_computsa,
        "n_obs_computsa": n1,
        "beta_houst": beta_houst,
        "alpha_houst": alpha_houst,
        "corr_houst": corr_houst,
        "n_obs_houst": n2,
    })

df_sku_betas = pd.DataFrame(results).merge(sku_lookup, on="sku_nbr", how="left")

print(df_sku_betas.head())

c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3015: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:2888: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\TobyWong\Desktop\UNCC\6211\advanced_business_analytics\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value enco

   sku_nbr  beta_computsa  alpha_computsa  corr_computsa  n_obs_computsa  \
0   110241       0.135968      158.137475       0.238997              37   
1   110269       0.048480      915.139494       0.034823              37   
2   110630      -0.228148     1476.495289      -0.166250              37   
3   110661      -0.038102      631.979692      -0.051361              37   
4   110689      -0.017911      638.241346      -0.021279              37   

   beta_houst  alpha_houst  corr_houst  n_obs_houst  \
0    0.023285   332.870769    0.031403           37   
1    0.108931   838.008309    0.060032           37   
2   -0.100652  1268.627127   -0.056273           37   
3    0.037478   522.042902    0.038760           37   
4    0.096653   477.048565    0.088098           37   

                             sku_name                 sub_class  \
0             4"X1-1/2" DWV WYE HXHXH  026-001-031-DWV FITTINGS   
1        4"X1-1/2" DWV SANI TEE HXHXH  026-001-031-DWV FITTINGS   
2  2" DWV S

In [46]:
df_sku_betas["best_driver"] = np.where(
    df_sku_betas["corr_computsa"].abs() >= df_sku_betas["corr_houst"].abs(),
    "COMPUTSA",
    "HOUST"
)

df_sku_betas["best_beta"] = np.where(
    df_sku_betas["best_driver"] == "COMPUTSA",
    df_sku_betas["beta_computsa"],
    df_sku_betas["beta_houst"]
)

df_sku_betas["best_corr"] = np.where(
    df_sku_betas["best_driver"] == "COMPUTSA",
    df_sku_betas["corr_computsa"],
    df_sku_betas["corr_houst"]
)

print(df_sku_betas[[
    "sku_nbr", "sku_name", "best_driver", "best_beta", "best_corr"
]].head())

   sku_nbr                            sku_name best_driver  best_beta  \
0   110241             4"X1-1/2" DWV WYE HXHXH    COMPUTSA   0.135968   
1   110269        4"X1-1/2" DWV SANI TEE HXHXH       HOUST   0.108931   
2   110630  2" DWV SOIL PIPE ADAPTER HX NO HUB    COMPUTSA  -0.228148   
3   110661  3" DWV SOIL PIPE ADAPTER HX NO HUB    COMPUTSA  -0.038102   
4   110689  4" DWV SOIL PIPE ADAPTER HX NO HUB       HOUST   0.096653   

   best_corr  
0   0.238997  
1   0.060032  
2  -0.166250  
3  -0.051361  
4   0.088098  


In [47]:
df_sku_betas.to_csv("data/correlation_coefficients.csv")